# 🍄 슈퍼마리오 DQN 학습
**런타임 → 런타임 유형 변경 → T4 GPU 선택 후 실행하세요**

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/supermario_dl_project'

import os
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results', exist_ok=True)
print('Drive 마운트 완료!')

## 2. 패키지 설치

In [ ]:
!pip install gym==0.21.0 gym-super-mario-bros==7.4.0 nes-py==8.2.1 opencv-python-headless tqdm -q
print('설치 완료!')

## 3. GitHub에서 프로젝트 클론

In [ ]:
# GitHub 레포 주소를 입력하세요
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/supermario_dl_project.git'

import os
if not os.path.exists('/content/supermario_dl_project'):
    !git clone {GITHUB_REPO} /content/supermario_dl_project
else:
    !cd /content/supermario_dl_project && git pull

os.chdir('/content/supermario_dl_project')
print('프로젝트 준비 완료!')

## 4. GPU 확인

In [ ]:
import torch
print(f'GPU 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 모델: {torch.cuda.get_device_name(0)}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

## 5. 학습 실행

In [ ]:
import sys
sys.path.insert(0, '/content/supermario_dl_project')

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.env.wrappers import make_env
from src.agent.dqn_agent import DQNAgent

# ====== 설정 ======
EPISODES = 3000          # 총 에피소드 수
SAVE_EVERY = 200         # 몇 에피소드마다 저장
CHECKPOINT_PATH = None   # 이어서 학습: f'{DRIVE_PATH}/models/mario_ep1000.pth'
# ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
env = make_env(world=1, stage=1)
n_actions = env.action_space.n
agent = DQNAgent(n_actions=n_actions, device=device)

start_episode = 0
if CHECKPOINT_PATH and os.path.exists(CHECKPOINT_PATH):
    agent.load(CHECKPOINT_PATH)
    start_episode = int(CHECKPOINT_PATH.split('ep')[-1].split('.')[0])
    print(f'에피소드 {start_episode}부터 재개')

rewards_history = []
best_reward = -float('inf')

for episode in tqdm(range(start_episode, start_episode + EPISODES)):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        action = agent.select_action(state)
        next_state, reward, done, info = env.step(action)
        agent.store(state, action, reward, next_state, done)
        agent.learn()
        state = next_state
        total_reward += reward

    rewards_history.append(total_reward)

    if (episode + 1) % SAVE_EVERY == 0:
        path = f'{DRIVE_PATH}/models/mario_ep{episode+1}.pth'
        agent.save(path)
        avg = np.mean(rewards_history[-100:])
        print(f'[EP {episode+1}] 평균보상: {avg:.1f} | epsilon: {agent.epsilon:.3f} | 저장완료')

    if total_reward > best_reward:
        best_reward = total_reward
        agent.save(f'{DRIVE_PATH}/models/mario_best.pth')

env.close()
print('학습 완료!')

## 6. 학습 곡선 저장

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(rewards_history, alpha=0.4, label='에피소드 보상')
window = min(100, len(rewards_history))
avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(rewards_history)), avg, label=f'{window}회 평균')
plt.xlabel('에피소드')
plt.ylabel('보상')
plt.title('슈퍼마리오 DQN 학습 곡선')
plt.legend()
plt.savefig(f'{DRIVE_PATH}/results/training_curve.png', dpi=150)
plt.show()
print(f'그래프 저장: {DRIVE_PATH}/results/training_curve.png')

## 7. 이어서 학습할 때 (세션 끊긴 후)

In [ ]:
# 세션 끊기고 재접속 후 아래 셀부터 실행하면 됩니다
# CHECKPOINT_PATH 를 마지막 저장된 파일 경로로 변경 후 5번 셀 재실행

import os
saved_models = sorted(os.listdir(f'{DRIVE_PATH}/models'))
print('저장된 모델 목록:')
for m in saved_models:
    print(f'  {DRIVE_PATH}/models/{m}')